![ai](https://www.emineo-education.fr/wp-content/uploads/2022/11/supdevinci-nantes.png)


<h4 style="text-align: left; color:#20a08d; font-size: 35px"><span><strong> Assurez la modération des contenus multimédia avec AWS</strong></span></h4>

<h4 style="text-align: left; color:#20a08d; font-size: 25px"><span><strong> Introduction
</strong></span></h4>

Le saviez-vous ? Les réseaux sociaux du groupe Meta Facebook et Instagram recueillent environ 2 milliards d'images de leurs utilisateurs tous les jours. Imaginez toute l'infrastructure informatique nécessaire pour traiter toutes ces données

![](https://github.com/archiducarmel/SupDeVinci_Developpement/releases/download/ia_ml_aws/fb.gif)

Afin de fournir des services intuitifs à leurs utilisateurs, plusieurs traitements sont réalisés sur chacune de ces images.

![](https://github.com/archiducarmel/SupDeVinci_Developpement/releases/download/ia_ml_aws/fb2.png)

Nous allons utiliser dans ce TP les services AWS pour réaliser quelques-uns de ces fonctionnalités. La finalité ultime consiste à développer une fonction de traitement qui recueille une image ou une vidéo en entrée, la modère afin de vérifier si son contenu est publiable, produit des sous-titres (dans le cas des vidéos) et fournit des hashtags issus de mots les plus représentatifs du contenu.

<h4 style="text-align: left; color:#20a08d; font-size: 25px"><span><strong> Workflow de traitement
</strong></span></h4>

Voici ci-dessous le procédé de traitement qui sera appliqué de bout-en-bout sur toute image présentée en entrée de la fonction de traitement.

![](https://github.com/archiducarmel/SupDeVinci_Developpement/releases/download/ia_ml_aws/aws_socialmedia.drawio.png)

<h4 style="text-align: left; color:#20a08d; font-size: 25px"><span><strong> Détection du type de fichier
</strong></span></h4>

La fonction `check_filetype` ci-dessous permet de déterminer le type (image ou vidéo) d'un fichier fourni en entrée.

In [ ]:
import os

import os

def check_filetype(filename):
    ext = os.path.splitext(filename)[-1].lower()
    if ext in ['.jpg', '.jpeg', '.png', '.tiff', '.bmp', '.gif']:
        return 'image'
    elif ext in ['.mp4', '.avi', '.mkv', '.mov']:
        return 'video'
    return None


<p style="text-align: left; font-size: 16px; color:#131fcf"><span>🖥️  Appelez la fonction <code style="text-align: left; font-size: 16px; color:#131fcf">check_filetype</code> sur la vidéo de test et l'image de test afin d'en détecter le type</span></p>

In [ ]:
TEST_VIDEO_FILE = "./assets/tuto_maquillage.mp4"
TEST_IMAGE_FILE = "./assets/selfie_with_johnny-depp.png"

video_type = check_filetype(TEST_VIDEO_FILE)
image_type = check_filetype(TEST_IMAGE_FILE)

print(f"Type détecté pour la vidéo : {video_type}")
print(f"Type détecté pour l'image : {image_type}")

<h4 style="text-align: left; color:#20a08d; font-size: 25px"><span><strong> Extraction d'une image de la vidéo
</strong></span></h4>

La fonction `extract_frame_video` ci-dessous permet d'extraire une image sous forme de tableau de pixels d'une vidéo à partir de la position de l'image dans la vidéo

In [ ]:
import cv2

#Extrait une image spécifique d'une vidéo.
def extract_frame_video(video_path, frame_id):
    cap = cv2.VideoCapture(video_path)
    cap.set(cv2.CAP_PROP_POS_FRAMES, frame_id)
    success, frame = cap.read()
    if success:
        frame_path = f"frame_{frame_id}.jpg"
        cv2.imwrite(frame_path, frame)
        cap.release()
        return frame_path
    cap.release()
    return None

<p style="text-align: left; font-size: 16px; color:#131fcf"><span>🖥️  Appelez la fonction <code style="text-align: left; font-size: 16px; color:#131fcf">extract_frame_video</code> sur la vidéo de test afin d'en extraire la première image de la vidéo, puis affichez cette image avec <code style="text-align: left; font-size: 16px; color:#131fcf">matplotlib</code></span></p>

In [ ]:
import matplotlib.pyplot as plt

# Extraire la première image de la vidéo de test
frame = extract_frame_video(TEST_VIDEO_FILE, 0)

# Vérifier si l'extraction a réussi
if frame is not None:
    # Convertir l'image de BGR à RGB (OpenCV utilise BGR par défaut)
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    
    # Afficher l'image
    plt.imshow(frame_rgb)
    plt.axis("off")  # Cacher les axes
    plt.title("Première image extraite de la vidéo")
    plt.show()
else:
    print("L'extraction de l'image a échoué.")

<p style="text-align: left; font-size: 16px; color:#131fcf"><span>🖥️  Appelez la fonction <code style="text-align: left; font-size: 16px; color:#131fcf">extract_frame_video</code> sur la vidéo de test afin d'en extraire la quatre vingt dix-neuvième image de la vidéo, puis affichez cette image avec <code style="text-align: left; font-size: 16px; color:#131fcf">matplotlib</code></span></p>

In [ ]:
import matplotlib.pyplot as plt

# Extraire la 99ᵉ image de la vidéo de test
frame_99 = extract_frame_video(TEST_VIDEO_FILE, 98)  # L'index commence à 0

# Vérifier si l'extraction a réussi
if frame_99 is not None:
    # Convertir l'image de BGR à RGB (OpenCV utilise BGR par défaut)
    frame_99_rgb = cv2.cvtColor(frame_99, cv2.COLOR_BGR2RGB)
    
    # Afficher l'image
    plt.imshow(frame_99_rgb)
    plt.axis("off")  # Cacher les axes
    plt.title("99ᵉ image extraite de la vidéo")
    plt.show()
else:
    print("L'extraction de l'image a échoué.")

<h4 style="text-align: left; color:#20a08d; font-size: 25px"><span><strong> Modération d'une image
</strong></span></h4>

La fonction `get_aws_session` ci-dessous permet de se connecter à une session AWS en utilisant les clés d'accès et clés secrètes.

In [ ]:
#!pip install boto3 python-dotenv 
#!pip install nltk

In [ ]:
import os, boto3
from dotenv import load_dotenv

def get_aws_session():
    return boto3.Session(
        aws_access_key_id=os.getenv("ACCESS_KEY"),
        aws_secret_access_key=os.getenv("SECRET_KEY")
    )

Passons maintenant au développement de la fonction `moderate_image`. Cette fonction prendra en entrée une image et renverra la liste des thèmes choquants présents dans l'image, s'il y'en a. 

<p style="text-align: left; font-size: 16px; color:#7a0f43"><span>❓ Quelle service AWS serait le plus indiqué pour réaliser ce traitement ?</span></p>

In [ ]:
Amazon Rekognition

<p style="text-align: left; font-size: 16px; color:#131fcf"><span>🖥️  Ecrivez le code dans la fonction  <strong>moderate_image</strong> permettant d'analyser une image et détecter les sujets de modération </span></p>

In [ ]:
import boto3


#Détecte du contenu nécessitant une modération dans une image en utilisant un service AWS spécifié.
def moderate_image(image_path, client):
    with open(image_path, 'rb') as image_file:
        response = client.detect_moderation_labels(Image={'Bytes': image_file.read()})
    return response.get('ModerationLabels', [])

<p style="text-align: left; font-size: 16px; color:#131fcf"><span>🖥️  Ecrivez le code permettant de tester la fonction  <strong>moderate_image</strong>. Pour ce faire : <ul style="text-align: left; font-size: 16px; color:#131fcf">
    <li>Instancier une session AWS avec vos clés</li>
    <li>Instancier le service AWS approprié pour ce traitement </li>
    <li>Appelez la fonction <code style="text-align: left; font-size: 16px; color:#131fcf">moderate_image</code> avec ce service comme argument afin de recueillir la liste potentielle des thèmes choquants</li>
    </ul> </span></p>

In [ ]:
TEST_IMAGE_FILE_1 = "./assets/haine.png"
TEST_IMAGE_FILE_2 = "./assets/vulgaire.png"
TEST_IMAGE_FILE_3 = "./assets/violence1.png"
TEST_IMAGE_FILE_4 = "./assets/no-violence1.png"

import boto3

# Remplacez ces valeurs par vos propres identifiants AWS
AWS_ACCESS_KEY = "votre_access_key"
AWS_SECRET_KEY = "votre_secret_key"
AWS_REGION = "us-east-1"  # Modifier si nécessaire

# Instancier une session AWS
session = boto3.Session(
    aws_access_key_id=AWS_ACCESS_KEY,
    aws_secret_access_key=AWS_SECRET_KEY,
    region_name=AWS_REGION
)

# Instancier le service AWS Rekognition
rekognition_client = session.client("rekognition")

# Tester la fonction moderate_image sur les fichiers définis
for image_path in [TEST_IMAGE_FILE_1, TEST_IMAGE_FILE_2, TEST_IMAGE_FILE_3, TEST_IMAGE_FILE_4]:
    print(f"Analyse de {image_path} ...")
    moderation_labels = moderate_image(image_path, rekognition_client)
    print(f"Thèmes choquants détectés : {moderation_labels}\n")


<h4 style="text-align: left; color:#20a08d; font-size: 25px"><span><strong> Production de sous-titres
</strong></span></h4>

La production de sous-titres à partir d'une vidéo s'appuiera sur la technologie speech-to-text d'AWS.

<div class="alert alert-info">
  <strong>BUCKET S3</strong><br><br> Au préalable, assurez-vous d'avoir créé un bucket S3 puisque la transcription speech-to-text nécessite que le fichier transcrit soit déposé dans un bucket S3
</div>

<p style="text-align: left; font-size: 16px; color:#131fcf"><span>🖥️  Ecrivez le code permettant d'instancier un client S3 puis de créer un bucket </span></p>

In [ ]:
import boto3
 #mettre les identifiants
AWS_ACCESS_KEY = "votre_access_key"
AWS_SECRET_KEY = "votre_secret_key"
AWS_REGION = "us-east-1" 

# Nom du bucket 
BUCKET_NAME = "mon-bucket-exemple-unique"

# Instancier le client S3
s3_client = boto3.client(
    "s3",
    aws_access_key_id=AWS_ACCESS_KEY,
    aws_secret_access_key=AWS_SECRET_KEY,
    region_name=AWS_REGION
)

# Créer un bucket
try:
    s3_client.create_bucket(
        Bucket=BUCKET_NAME,
        CreateBucketConfiguration={"LocationConstraint": AWS_REGION}
    )
    print(f"Bucket '{BUCKET_NAME}' créé avec succès !")
except Exception as e:
    print(f"Erreur lors de la création du bucket : {e}")

<p style="text-align: left; font-size: 16px; color:#7a0f43"><span>❓ Quelle service AWS serait le plus indiqué pour réaliser ce traitement de transcription speech-to-text ?</span></p>

In [ ]:
Amazon Transcribe

<p style="text-align: left; font-size: 16px; color:#131fcf"><span>🖥️  Ecrivez le code de la fonction <code style="text-align: left; font-size: 16px; color:#131fcf">get_text_from_speech</code> permettant de réaliser la transcription speech-to-text avec AWS</span></p>

<p style="text-align: left; font-size: 16px; color:#ec8f1a"><span>📚  Voice to text using AWS Transcribe : </span> <a href="https://dev.to/botreetechnologies/voice-to-text-using-aws-transcribe-with-python-1cfc">https://dev.to/botreetechnologies/voice-to-text-using-aws-transcribe-with-python-1cfc</a></p> 

In [ ]:
#Convertit de la parole en texte en utilisant AWS Transcribe.
def get_text_from_speech(filename, transcribe, job_name, bucket_name):
    """
    Lance la transcription AWS Transcribe et attend la fin du job pour récupérer la transcription.
    """
    job_uri = f's3://{bucket_name}/{filename}'
    
    transcribe.start_transcription_job(
        TranscriptionJobName=job_name,
        Media={'MediaFileUri': job_uri},
        MediaFormat='mp4',
        LanguageCode='fr-FR'
    )
    
    # Attendre que le job soit complété
    while True:
        status = transcribe.get_transcription_job(TranscriptionJobName=job_name)
        if status['TranscriptionJob']['TranscriptionJobStatus'] in ['COMPLETED', 'FAILED']:
            break
        print("⏳ En attente de la transcription...")
        time.sleep(10)

    if status['TranscriptionJob']['TranscriptionJobStatus'] == 'COMPLETED':
        transcript_url = status['TranscriptionJob']['Transcript']['TranscriptFileUri']
        response = requests.get(transcript_url)
        transcript_data = response.json()

        if 'results' in transcript_data and 'transcripts' in transcript_data['results']:
            return transcript_data['results']['transcripts'][0]['transcript']
    
    return None


<p style="text-align: left; font-size: 16px; color:#131fcf"><span>🖥️  Ecrivez le code permettant de tester la fonction  <strong>get_text_from_speech</strong>. Pour ce faire : <ul style="text-align: left; font-size: 16px; color:#131fcf">
    <li>Uploader la vidéo de test sur le bucket de test préalablement créé</li>
    <li>Instancier le service AWS approprié pour ce traitement </li>
    <li>Appelez la fonction <code style="text-align: left; font-size: 16px; color:#131fcf">get_text_from_speech</code> avec ce service comme argument afin de recueillir le texte recueilli</li>
    </ul> </span></p>

In [ ]:
import boto3
import time
import urllib.request
import json

# Remplacez ces valeurs par vos propres identifiants AWS
AWS_ACCESS_KEY = "votre_access_key"
AWS_SECRET_KEY = "votre_secret_key"
AWS_REGION = "us-east-1"  # Modifier si nécessaire

# Nom du bucket et fichier à analyser
BUCKET_NAME = "test-bucket-sdvnantes-2"
TEST_VIDEO_FILE = "./assets/tuto_coiffure.mp4"  # Modifier avec votre fichier
S3_VIDEO_KEY = "test_video.mp4"  # Nom sous lequel stocker la vidéo sur S3
JOB_NAME = "transcription-job-test"

# Instancier une session AWS
session = boto3.Session(
    aws_access_key_id=AWS_ACCESS_KEY,
    aws_secret_access_key=AWS_SECRET_KEY,
    region_name=AWS_REGION
)

# Instancier le client S3
s3_client = session.client("s3")

# Uploader la vidéo sur S3
try:
    s3_client.upload_file(TEST_VIDEO_FILE, BUCKET_NAME, S3_VIDEO_KEY)
    print(f"Vidéo {TEST_VIDEO_FILE} uploadée sur S3 avec succès !")
except Exception as e:
    print(f"Erreur lors de l'upload de la vidéo : {e}")

# Instancier le client AWS Transcribe
transcribe_client = session.client("transcribe")

# Appeler la fonction pour tester
transcribed_text = get_text_from_speech(S3_VIDEO_KEY, transcribe_client, JOB_NAME, BUCKET_NAME)
print("Texte transcrit :", transcribed_text)


<h4 style="text-align: left; color:#20a08d; font-size: 25px"><span><strong> Production de hashtags d'une séquence vidéo
</strong></span></h4>

La production de hashtag sur une séquence vidéo se base sur le texte extrait de la vidéo après l'étape de speech-to-text, qui sera utilisé pour en extraire des mots-clés (keyphrases). Au préalable, le texte extrait devra être nettoyé pour y enlever quelques éléments inutiles. C'est la fonction de la fonction `clean_text`

In [ ]:
import nltk
nltk.download('stopwords')

from nltk.corpus import stopwords

#Nettoie un texte en retirant les mots vides et en normalisant les mots en minuscules.
def clean_text(raw_text):
    nltk.download('stopwords')
    stop_words = set(stopwords.words('french'))
    words = raw_text.lower().split()
    return ' '.join([word for word in words if word not in stop_words])

# Test de la fonction
texte_brut = "Ceci est un exemple de texte à nettoyer."
texte_nettoye = clean_text(texte_brut)
print("Texte nettoyé :", texte_nettoye)


<p style="text-align: left; font-size: 16px; color:#131fcf"><span>🖥️  Appelez la fonction <code style="text-align: left; font-size: 16px; color:#131fcf">clean_text</code> le texte extrait afin de recueillir un texte nettoyé</span></p>

In [ ]:
# Supposons que text_extrait soit le texte récupéré avec get_text_from_speech
text_extrait = "Ceci est un exemple de transcription de texte obtenue à partir d'un fichier audio."

# Appel de la fonction clean_text pour nettoyer le texte
text_nettoye = clean_text(text_extrait)

# Affichage du texte nettoyé
print("Texte nettoyé :", text_nettoye)


<p style="text-align: left; font-size: 16px; color:#7a0f43"><span>❓ Quelle service AWS serait le plus indiqué pour réaliser ce traitement d'extraction des "key phrases" ?</span></p>

In [ ]:
Amazon Comprehend

<p style="text-align: left; font-size: 16px; color:#131fcf"><span>🖥️  Ecrivez le code de la fonction <code style="text-align: left; font-size: 16px; color:#131fcf">extract_keyphrases</code> permettant d'extraire les mots clés d'un texte en entrée. Ne retenez que les 10 mots-clés détectés avec le plus de confiance</span></p>

In [ ]:
#Extrait les expressions clés d'un texte et retourne les 10 expressions les plus pertinentes comme hashtags.
def extract_keyphrases(text, comprehend):
    response = comprehend.detect_key_phrases(Text=text, LanguageCode='fr')
    keyphrases = [phrase['Text'] for phrase in response['KeyPhrases']]
    return keyphrases[:10]

<p style="text-align: left; font-size: 16px; color:#131fcf"><span>🖥️  Ecrivez le code permettant de tester la fonction  <strong>extract_keyphrases</strong>. Pour ce faire : <ul style="text-align: left; font-size: 16px; color:#131fcf">
    <li>Instancier le service AWS approprié pour ce traitement </li>
    <li>Appelez la fonction <code style="text-align: left; font-size: 16px; color:#131fcf">extract_keyphrases</code> avec ce service comme argument afin de recueillir la liste des mots-clés</li>
    </ul> </span></p>

In [ ]:
import boto3

# vos acces
AWS_ACCESS_KEY = "votre_access_key"
AWS_SECRET_KEY = "votre_secret_key"
AWS_REGION = "us-east-1" 

# Instancier une session AWS
session = boto3.Session(
    aws_access_key_id=AWS_ACCESS_KEY,
    aws_secret_access_key=AWS_SECRET_KEY,
    region_name=AWS_REGION
)

# Instancier le service AWS Comprehend
comprehend_client = session.client("comprehend")

# Texte à analyser
test_text = "Le festival Electro'Loire propose une fusion entre musique électronique et patrimoine nantais."

# Tester la fonction extract_keyphrases
hashtags = extract_keyphrases(test_text, comprehend_client)

# Afficher les résultats
print("Hashtags générés :", hashtags)


<h4 style="text-align: left; color:#20a08d; font-size: 25px"><span><strong> Production de hashtags d'une image
</strong></span></h4>

La production de hashtags sur une image se base sur la détection des objets et des célébrités présents dans l'image.

<h4 style="text-align: left; color:#20a08d; font-size: 20px"><span><strong> Détection d'objets sur une image
</strong></span></h4>

<p style="text-align: left; font-size: 16px; color:#131fcf"><span>🖥️  Ecrivez le code de la fonction <code style="text-align: left; font-size: 16px; color:#131fcf">detect_objects</code> permettant de détecter les objets présents sur une image donnée en entrée de la fonction. Ne retenez que les 10 objets détectés avec le plus de confiance.</span></p>

In [ ]:
#Détecte les objets dans une image en utilisant Amazon Rekognition.
def detect_objects(image_path, client):
    with open(image_path, 'rb') as image_file:
        response = client.detect_labels(Image={'Bytes': image_file.read()}, MaxLabels=10)
    return [label['Name'] for label in response.get('Labels', [])]


<p style="text-align: left; font-size: 16px; color:#131fcf"><span>🖥️  Ecrivez le code permettant de tester la fonction  <strong>detect_objects</strong>. Pour ce faire : <ul style="text-align: left; font-size: 16px; color:#131fcf">
    <li>Instancier le service AWS approprié pour ce traitement </li>
    <li>Appelez la fonction <code style="text-align: left; font-size: 16px; color:#131fcf">detect_objects</code> avec ce service comme argument afin de recueillir la liste des objets présents sur cette image de test</li>
    </ul> </span></p>

In [ ]:
import boto3

AWS_ACCESS_KEY = "votre_access_key"
AWS_SECRET_KEY = "votre_secret_key"
AWS_REGION = "us-east-1"  # Modifier si nécessaire

# Instancier une session AWS
session = boto3.Session(
    aws_access_key_id=AWS_ACCESS_KEY,
    aws_secret_access_key=AWS_SECRET_KEY,
    region_name=AWS_REGION
)

# Instancier le client AWS Rekognition
rekognition_client = session.client("rekognition")

# Spécifier le chemin de l'image à analyser
TEST_IMAGE_PATH = "./assets/no-violence4.png"

# Tester la fonction detect_objects
detected_objects = detect_objects(TEST_IMAGE_PATH, rekognition_client)

# Afficher les résultats
print("Objets détectés :", detected_objects)


<h4 style="text-align: left; color:#20a08d; font-size: 20px"><span><strong> Détection des célébrités sur une image
</strong></span></h4>

<p style="text-align: left; font-size: 16px; color:#131fcf"><span>🖥️  Ecrivez le code de la fonction <code style="text-align: left; font-size: 16px; color:#131fcf">detect_celebrities</code> permettant de détecter les célébrités présents sur une image donnée en entrée de la fonction.</span></p>

In [ ]:
def detect_celebrities(image_path, aws_service):
    """
    Identifie les célébrités dans une image en utilisant le service Amazon Rekognition.

    Cette fonction ouvre une image depuis un chemin donné et utilise le service AWS Rekognition pour reconnaître les
    célébrités présentes dans l'image. Elle retourne une liste contenant les noms des célébrités identifiées, limitée
    aux 10 premiers résultats pour simplifier l'output.

    Paramètres :
    - image_path (str) : Le chemin vers l'image dans laquelle détecter les célébrités.
    - aws_service (object) : Un client AWS Rekognition configuré.

    Retourne :
    - list[str] : Une liste des noms des célébrités identifiées dans l'image, jusqu'à un maximum de 10.
    """

    with open(image_path, "rb") as image_file:
        image_bytes = image_file.read()

    response = aws_service.recognize_celebrities(Image={"Bytes": image_bytes})

    celebrities = [celeb["Name"] for celeb in response.get("CelebrityFaces", [])][:10]

    return celebrities


<p style="text-align: left; font-size: 16px; color:#131fcf"><span>🖥️  Ecrivez le code permettant de tester la fonction  <strong>detect_celebrities</strong>. Pour ce faire : <ul style="text-align: left; font-size: 16px; color:#131fcf">
    <li>Instancier le service AWS approprié pour ce traitement </li>
    <li>Appelez la fonction <code style="text-align: left; font-size: 16px; color:#131fcf">detect_celebrities</code> avec ce service comme argument afin de recueillir la liste des célébrités présentes sur chacune des images de test</li>
    </ul> </span></p>

In [ ]:
import boto3

# Définition des images de test
TEST_IMAGE_FILE_1 = "./assets/selfie_with_mariah-carey.png"
TEST_IMAGE_FILE_2 = "./assets/selfie_with_johnny-depp.png"
TEST_IMAGE_FILE_3 = "./assets/selfie_with_kanye-west.png"

# Configuration des identifiants AWS (Remplacez par vos propres credentials)
AWS_ACCESS_KEY = "votre_access_key"
AWS_SECRET_KEY = "votre_secret_key"
AWS_REGION = "us-east-1"  # Modifier si nécessaire

# Instanciation d'un client AWS Rekognition
session = boto3.Session(
    aws_access_key_id=AWS_ACCESS_KEY,
    aws_secret_access_key=AWS_SECRET_KEY,
    region_name=AWS_REGION
)
rekognition_client = session.client("rekognition")

# Fonction pour tester detect_celebrities sur plusieurs images
def test_detect_celebrities():
    test_images = [TEST_IMAGE_FILE_1, TEST_IMAGE_FILE_2, TEST_IMAGE_FILE_3]
    
    for image_path in test_images:
        celebrities = detect_celebrities(image_path, rekognition_client)
        print(f"Célébrités détectées dans {image_path}: {celebrities}")

# Exécution du test
test_detect_celebrities()


<h4 style="text-align: left; color:#20a08d; font-size: 20px"><span><strong> Reconnaissance d'émotion faciale sur une image
</strong></span></h4>

<span style="color:#131fcf">🖥️ Codez la fonction `detect_emotions` qui doit :

<ul style="color:#131fcf">
<li>Prendre en entrée :
  <ul>
    <li>Le chemin de l'image à analyser</li>
    <li>Le client AWS Rekognition configuré</li>
  </ul>
</li>

<li>Analyser l'image :
  <ul>
    <li>Ouvrir l'image en mode binaire</li>
    <li>Utiliser Rekognition avec <strong>detect_faces</strong></li>
    <li>Demander tous les attributs (Attributes=['ALL'])</li>
  </ul>
</li>

<li>Pour chaque visage détecté, afficher :
  <ul>
    <li>Le genre avec son niveau de confiance</li>
    <li>L'âge estimé (range min-max)</li>
    <li>Les 3 émotions principales avec leur niveau de confiance</li>
  </ul>
</li>

<li>Retourner la liste complète des informations des visages détectés</li>

<li>Exemple de sortie console attendue :
<code style="color:#131fcf">
[INFO] Visage détecté:
  - Genre: Male (confiance: 99.9%)
  - Âge estimé: 20-30 ans
  - Émotions principales:
    * HAPPY: 95.5%
    * CALM: 4.5%
---
</code>
</li>
</ul>
</span>

In [ ]:
#Détecte les émotions sur les visages présents dans une image en utilisant Amazon Rekognition.
def detect_emotions(image_path, client):
    with open(image_path, 'rb') as image_file:
        response = client.detect_faces(Image={'Bytes': image_file.read()}, Attributes=['ALL'])
    return response.get('FaceDetails', [])

<span style="color:#131fcf">🖥️ Codez la fonction `summarize_emotions` qui doit :

<ul style="color:#131fcf">
<li>Prendre en entrée une liste de visages détectés dans une image comme fourni par la fonction <code>detect_emotions</code></li>
<li>Exemple d'entrée :
<code style="color:#131fcf">
[{
    'Gender': {'Value': 'Male', 'Confidence': 99.9},
    'AgeRange': {'Low': 20, 'High': 30},
    'Emotions': [
        {'Type': 'HAPPY', 'Confidence': 95.5},
        {'Type': 'CALM', 'Confidence': 4.5}
    ]
}]
</code>
<li>Pour chaque visage, analyser :
  <ul>
    <li>Le genre (Homme/Femme)</li>
    <li>L'âge (calcul de la moyenne du range)</li>
    <li>Les émotions avec une confiance > 50%</li>
  </ul>
</li>

<li>Retourner un dictionnaire avec :
  <ul>
    <li>Nombre total de visages</li>
    <li>Émotion dominante (celle avec la plus haute confiance moyenne)</li>
    <li>Statistiques des émotions (comptage et confiance moyenne)</li>
    <li>Statistiques d'âge (min, max, moyenne)</li>
    <li>Distribution des genres</li>
  </ul>
</li>
</li>
</ul>
</span>

In [ ]:
#Détecte les émotions sur les visages présents dans une image en utilisant Amazon Rekognition.
def detect_emotions(image_path, client):
    with open(image_path, 'rb') as image_file:
        response = client.detect_faces(Image={'Bytes': image_file.read()}, Attributes=['ALL'])
    return response.get('FaceDetails', [])

<ul style="color:#131fcf">
<li>Testez la détection et l'analyse d'émotions :
  <ul>
    <li>Sur chacune des 4 images de groupe</li>
    <li>Comparez les résultats entre elles</li>
  </ul>
</li>
<li>Pour chaque image :
  <ul>
    <li>Afficher les détails de chaque visage détecté</li>
    <li>Générer le résumé des statistiques</li>
    <li>Noter les différences d'émotions dominantes</li>
  </ul>
</li>
</li>
</ul>
</span>

In [ ]:
import boto3

# Définir les chemins des images de test
TEST_IMAGE_FILES = [
    "./assets/group_selfie_1.jpg",
    "./assets/group_selfie_2.jpg",
    "./assets/group_selfie_3.jpg",
    "./assets/group_selfie_4.jpg"
]

# Initialiser le client AWS Rekognition
rekognition_client = boto3.client('rekognition', region_name='us-east-1')

# Tester la détection et l'analyse des émotions sur chaque image
for image_path in TEST_IMAGE_FILES:
    print(f"\n📸 Analyse des émotions pour : {image_path}")

    # Détecter les visages et leurs émotions
    detected_faces = detect_emotions(image_path, rekognition_client)

    # Afficher les détails de chaque visage détecté
    for i, face in enumerate(detected_faces, 1):
        print(f"\n👤 Visage {i} :")
        for emotion in face['Emotions']:
            print(f"  - {emotion['Type']}: {emotion['Confidence']:.2f}%")

    # Générer le résumé des émotions pour l'image
    summary = summarize_emotions(detected_faces)
    print("\n📊 Résumé des émotions :")
    print(f"  - Émotion dominante : {summary['dominant_emotion']}")
    print(f"  - Détails : {summary['emotions_summary']}")

    print("\n" + "="*50)


<h4 style="text-align: left; color:#20a08d; font-size: 20px"><span><strong> Fonction de traitement finale
</strong></span></h4>

Il est maintenant temps de développer la fonction de traitement finale `process_media` qui se basera sur l'ensemble des fonctions développées précédemment.

<p style="text-align: left; font-size: 16px; color:#131fcf"><span>🖥️  Ecrivez le code de la fonction <code style="text-align: left; font-size: 16px; color:#131fcf">process_media</code> permettant de réaliser l'ensemble des traitements : <ul style="text-align: left; font-size: 16px; color:#131fcf">
    <li>Déterminer le type de média (vidéo ou image)</li>
    <li>Si le média est une image : </li>
    <ul style="text-align: left; font-size: 16px; color:#131fcf">
        <li>Modérer l'image</li>
        <li>Si aucun contenu choquant n'est détecté,  détecter les objets, l'émotion dominante des visages et les célébrités présents sur l'image qui serviront de mot-clés pour produire les hashtags</li>
        <li>Si du contenu choquant est trouvé, retourner <strong>None</strong></li>
    </ul>
    <li>Si le média est une vidéo : </li>
    <ul style="text-align: left; font-size: 16px; color:#131fcf">
        <li>Extraire la première image de la vidéo</li>
        <li>Sauvegarder cette image comme fichier temporaire</li>
        <li>Modérer cette première image</li>
        <li>Si aucun contenu choquant n'est détecté sur cette image,  convertir la voix présente sur la vidéo en texte</li>
        <li>Extraire les mots-clés du texte extrait</li>
        <li>Si du contenu choquant est trouvé, retourner <strong>None</strong></li>
    </ul>
    <li>La sortie de cette fonction devra être un dictionnaire et avoir ce format : <strong>{subtitles : "abcdefgijklm", hashtags:["hastag1", "hastag1", ...]}</strong> pour une vidéo et <strong>{hashtags:["hastag1", "hastag1", ...]} pour une image</strong> </li>
</ul></span></p>

In [ ]:
def process_media(media_file, rekognition, transcribe, comprehend, s3, bucket_name):
    file_type = check_filetype(media_file)

    # 🔍 Gestion des images
    if file_type == 'image':
        moderation_labels = moderate_image(media_file, rekognition)
        objects = detect_objects(media_file, rekognition)
        celebrities = detect_celebrities(media_file, rekognition)
        emotions = detect_emotions(media_file, rekognition)

        return {
            'moderation': moderation_labels,
            'objects': objects,
            'celebrities': celebrities,
            'emotions': summarize_emotions(emotions)
        }

    # 🎥 Gestion des vidéos
    elif file_type == 'video':
        s3.upload_file(media_file, bucket_name, os.path.basename(media_file))

        # Vérification de l'upload
        uploaded_objects = s3.list_objects_v2(Bucket=bucket_name)
        uploaded_files = [obj['Key'] for obj in uploaded_objects.get('Contents', [])]

        if os.path.basename(media_file) not in uploaded_files:
            raise ValueError(f"⚠️ Problème lors de l'upload, fichier {media_file} introuvable dans S3 !")
        else:
            print(f"✅ Fichier {media_file} bien uploadé sur S3.")

        job_name = os.path.splitext(os.path.basename(media_file))[0]
        transcription_text = get_text_from_speech(os.path.basename(media_file), transcribe, job_name, bucket_name)

        if transcription_text:
            cleaned_text = clean_text(transcription_text)
            keyphrases = extract_keyphrases(cleaned_text, comprehend)
        else:
            cleaned_text = "⚠️ Échec de la transcription"
            keyphrases = []

        # 🖼 Extraire une image d'un frame de la vidéo
        extracted_frame = extract_frame_video(media_file, 100)  # Frame à 100ms

        # 🎯 Analyse du frame pour extraire des informations visuelles
        if extracted_frame:
            frame_analysis = process_media(extracted_frame, rekognition, transcribe, comprehend, s3, bucket_name)
            objects = frame_analysis.get('objects', [])
            celebrities = frame_analysis.get('celebrities', [])
            emotions = frame_analysis.get('emotions', [])
        else:
            objects, celebrities, emotions = [], [], []

        return {
            'transcription': cleaned_text,
            'keyphrases': keyphrases,
            'objects': objects,
            'celebrities': celebrities,
            'emotions': emotions
        }

    return None



<p style="text-align: left; font-size: 16px; color:#131fcf"><span>🖥️  Ecrivez le code permettant de tester la fonction  <strong>process_media</strong>. Pour ce faire : <ul style="text-align: left; font-size: 16px; color:#131fcf">
        <li>Instancier une session AWS avec vos clés</li>
    <li>Instancier les services AWS appropriés pour tous les traitements </li>
    <li>Appelez la fonction <code style="text-align: left; font-size: 16px; color:#131fcf">process_media</code> sur l'image de test et la vidéo de test afin d'en vérifier le bon fonctionnement </li>
    </ul> </span></p>

In [ ]:
import boto3
import json
from process_media import process_media  # Importer la fonction process_media depuis votre fichier

# 🔹 Chemins des fichiers de test
TEST_VIDEO_FILE = "./assets/tuto_jeux-video.mp4"
TEST_IMAGE_FILE = "./assets/selfie_with_johnny-depp.png"
BUCKET_NAME = 'test-bucket-sdvnantes-2'

aws_session = boto3.Session(
    aws_access_key_id="VOTRE_AWS_ACCESS_KEY",
    aws_secret_access_key="VOTRE_AWS_SECRET_ACCESS_KEY",
    region_name="us-east-1" 
)

rekognition_client = aws_session.client("rekognition")
transcribe_client = aws_session.client("transcribe")
comprehend_client = aws_session.client("comprehend")
s3_client = aws_session.client("s3")

print("\n🖼️ Analyse de l'image...")
image_result = process_media(TEST_IMAGE_FILE, rekognition_client, transcribe_client, comprehend_client, BUCKET_NAME)
print("🔹 Résultat de l'analyse de l'image :", json.dumps(image_result, indent=2, ensure_ascii=False))

print("\n🎥 Analyse de la vidéo...")
video_result = process_media(TEST_VIDEO_FILE, rekognition_client, transcribe_client, comprehend_client, BUCKET_NAME)
print("🔹 Résultat de l'analyse de la vidéo :", json.dumps(video_result, indent=2, ensure_ascii=False))



<h4 style="text-align: left; color:#20a08d; font-size: 25px"><span><strong> Resources 📚📚</strong></span></h4>

* <a href="https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/translate.html" target="_blank">Translate with Boto3</a>
* <a href="https://boto3.amazonaws.com/v1/documentation/api/latest/reference/services/textract.html#Textract.Client.start_document_text_detection" target="_blank">Textract Documentation</a>
* <a href="https://aws.amazon.com/textract/" target="_blank">Textract Landing</a>

### Réponse aux questions sur les services AWS

**🔹 Quel service AWS serait le plus indiqué pour réaliser le traitement d'images ?**  
👉 Réponse : `Amazon Rekognition`

**🔹 Quel service AWS serait le plus indiqué pour réaliser le traitement de transcription speech-to-text ?**  
👉 Réponse : `Amazon Transcribe`

**🔹 Quel service AWS serait le plus indiqué pour réaliser le traitement d'extraction des "key phrases" ?**  
👉 Réponse : `Amazon Comprehend`

In [ ]:
import boto3

# Initialiser le client Rekognition
rekognition_client = boto3.client('rekognition')

# Exemple d'analyse d'une image stockée dans S3
response = rekognition_client.detect_labels(
    Image={'S3Object': {'Bucket': 'mon-bucket', 'Name': 'image.jpg'}},
    MaxLabels=10
)

print(response)

In [ ]:
import boto3

# Initialiser le client Transcribe
transcribe_client = boto3.client('transcribe')

# Démarrer une transcription à partir d'un fichier audio S3
response = transcribe_client.start_transcription_job(
    TranscriptionJobName='MonJobTranscription',
    LanguageCode='fr-FR',
    Media={'MediaFileUri': 's3://mon-bucket/audio.mp3'}
)

print(response)

In [ ]:
import boto3

# Initialiser le client Comprehend
comprehend_client = boto3.client('comprehend')

# Analyser un texte pour extraire les phrases clés
response = comprehend_client.detect_key_phrases(
    Text="AWS offre une large gamme de services cloud.",
    LanguageCode='fr'
)

print(response)